# Regime Detection Quickstart

Runnable demo of the **Financial Dynamics** Bayesian regime-classification pipeline on **synthetic** OHLCV.

## Important (please read)

- **Not investment advice.** This notebook is an engineering / research demonstration of a citeable Python library. It does not recommend trades, allocations, or securities.
- **No live or backtest performance claims.** Any counts or recovery rates below are measured against **synthetic labeled segments** generated in this notebook. They are not historical trading results and are not evidence of predictive edge.
- **Dependencies:** core package extras only (`numpy`, `pandas`, `matplotlib`, and the installed `financial-dynamics` package). No network calls.

Install from the repo root if needed:

```bash
pip install -e ".[dev]"
# or: pip install financial-dynamics
```


## 1. Imports and synthetic data

We generate four engineered regime segments (Calm Trend → Volatile Trend → Chop → Risk-Off) using the same helper as the unit tests. Labels exist only because the data is synthetic.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Editable install puts the package on sys.path; fall back for bare clones.
ROOT = Path.cwd().resolve()
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
    sys.path.insert(0, str(ROOT / "scripts"))
elif (ROOT.parent / "src").is_dir():
    # Notebook opened from notebooks/
    sys.path.insert(0, str(ROOT.parent / "src"))
    sys.path.insert(0, str(ROOT.parent / "scripts"))

from financial_dynamics import REGIME_NAMES, FinancialDynamicsPipeline, Regime
from generate_synthetic_data import generate_synthetic_ohlcv

df, labels = generate_synthetic_ohlcv(seed=42, n_cycles=2)
print(f"Bars: {len(df)} | columns: {list(df.columns)}")
print("True (synthetic) label counts:")
print(labels.value_counts().to_string())
df.head()


## 2. Run the pipeline

`FinancialDynamicsPipeline.run` walks all five phases: features → centroid probabilities → Bayesian transitions → stabilization → risk overlays.


In [ ]:
pipeline = FinancialDynamicsPipeline()
results = pipeline.run(df)

valid = results.dropna(subset=["risk_adjusted_regime"])
print(f"Warmup bars: {pipeline.warmup_bars}")
print(f"Classified:  {len(valid)} / {len(results)}")
print()
print("Inferred regime distribution (synthetic series):")
print(valid["risk_adjusted_regime"].value_counts().to_string())
valid[["risk_adjusted_regime"] + [c for c in valid.columns if c.startswith("post_prob_")]].tail(3)


## 3. Synthetic label recovery (demo only)

Compare inferred `risk_adjusted_regime` to the synthetic ground-truth labels **after warmup**. This is a recovery check on constructed data — **not** a historical backtest metric and **not** a claim of real-market accuracy.


In [ ]:
aligned = valid.join(labels.rename("true_regime"), how="inner")
# Both sides use Regime enum names (e.g. CALM_TREND)
pred = aligned["risk_adjusted_regime"].astype(str)
true = aligned["true_regime"].astype(str)

mask = pred.notna() & true.notna()
n = int(mask.sum())
hits = int((pred[mask] == true[mask]).sum())
recovery = hits / n if n else float("nan")

print(f"Synthetic recovery (post-warmup): {hits}/{n} = {recovery:.1%}")
print()
print("Confusion (rows=true, cols=predicted) — synthetic only:")
print(pd.crosstab(true[mask], pred[mask]).to_string())
print()
print(
    "Reminder: synthetic recovery is for pipeline smoke / pedagogy. "
    "Do not cite this figure as live or historical performance."
)


## 4. Forecast and transition matrix

After a batch run the pipeline can emit a short-horizon regime forecast from the learned transition matrix.


In [ ]:
forecast = pipeline.forecast(horizon=10)
if forecast is None:
    print("Forecast unavailable (pipeline not warmed / no posterior yet).")
else:
    print(f"Expected duration in current regime: {forecast.expected_duration:.1f} bars")
    path = " → ".join(REGIME_NAMES[r] for r in forecast.most_likely_path[:5])
    print(f"Most likely path (first 5): {path}")

tm = pipeline._transition_engine.get_transition_matrix()
tm_df = pd.DataFrame(
    tm,
    index=[REGIME_NAMES[r] for r in Regime],
    columns=[REGIME_NAMES[r] for r in Regime],
)
print()
print("Learned transition matrix (rows=from, cols=to):")
tm_df.round(3)


## 5. Price path with inferred regimes

Simple close-price plot colored by inferred regime. Cosmetic only — not a trading chart.


In [ ]:
plot_df = valid.copy()
plot_df["close"] = df.loc[plot_df.index, "close"]

# risk_adjusted_regime stores Regime enum *names* (e.g. CALM_TREND)
colors = {
    "CALM_TREND": "#2a9d8f",
    "VOLATILE_TREND": "#e9c46a",
    "CHOP": "#7b2cbf",
    "RISK_OFF": "#e76f51",
}

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(plot_df.index, plot_df["close"], color="#264653", linewidth=1.2, label="close")

regime = plot_df["risk_adjusted_regime"].astype(str)
starts = regime.ne(regime.shift(fill_value=regime.iloc[0]))
segment_ids = starts.cumsum()
for _, seg in plot_df.groupby(segment_ids):
    rname = str(seg["risk_adjusted_regime"].iloc[0])
    ax.axvspan(seg.index[0], seg.index[-1], color=colors.get(rname, "#cccccc"), alpha=0.25)

ax.set_title("Synthetic price with inferred regimes (demo)")
ax.set_ylabel("Price")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()


## Next steps

- Script counterparts: [`examples/quickstart.py`](../examples/quickstart.py), [`examples/backtesting.py`](../examples/backtesting.py) (also synthetic).
- Chronology-safe research notes: [`docs/research/market-regime-temporal-validation.md`](../docs/research/market-regime-temporal-validation.md).
- Interactive public demo: [financial-dynamics-model.streamlit.app](https://financial-dynamics-model.streamlit.app) (`streamlit run app.py`).
- Portfolio framing (null/negative honesty, no predictive-edge claim): the Micap quant research portfolio indexes this library as a descriptive regime/risk characterization — not a live trading system.

**Disclaimer again:** educational / research software under MIT. Not investment advice. Synthetic demo metrics are not historical performance.
